# CropHist API Quickstart

This notebook is the engineering path for evaluating the CropHist API.

It uses field IDs from the published Iowa sample and shows how to:

1. prepare a batch of sample field IDs
2. call `POST /v1/fields`
3. flatten the JSON response into an analyst-friendly dataframe
4. inspect one raw API record and one pivoted history table

You need a valid `CROPHIST_API_KEY` for the request cells.


In [ ]:
import os
from itertools import islice

import pandas as pd
import requests
from IPython.display import display

SAMPLE_BASE = 'https://crophist.com/pages/iowa-sample/downloads'
API_BASE = os.environ.get('CROPHIST_API_BASE', 'https://api.crophist.com')
API_KEY = os.environ.get('CROPHIST_API_KEY', '').strip()
BATCH_SIZE = 20

if not API_KEY:
    print('Set CROPHIST_API_KEY before running the API request cells.')
    print("In Colab you can run: import os; os.environ['CROPHIST_API_KEY'] = '...'; then rerun this cell.")


In [ ]:
field_lookup = (
    pd.read_parquet(
        f'{SAMPLE_BASE}/iowa-fields-sample-long.parquet',
        columns=['field_id', 'lon', 'lat'],
    )
    .drop_duplicates()
    .sort_values('field_id')
    .head(BATCH_SIZE)
    .reset_index(drop=True)
)

FIELD_IDS = field_lookup['field_id'].tolist()
print(f'Prepared {len(FIELD_IDS)} Iowa sample field IDs for the API quickstart.')
display(field_lookup)


In [ ]:
def chunks(items, size):
    iterator = iter(items)
    while True:
        batch = list(islice(iterator, size))
        if not batch:
            return
        yield batch


def fetch_histories(field_ids):
    headers = {
        'Authorization': f'Bearer {API_KEY}',
        'Content-Type': 'application/json',
    }
    rows = []
    for batch in chunks(field_ids, 100):
        response = requests.post(
            f'{API_BASE}/v1/fields',
            headers=headers,
            json={'field_ids': batch},
            timeout=30,
        )
        if response.status_code == 403:
            print('CropHist API returned 403 Forbidden. Check CROPHIST_API_KEY and rerun this cell.')
            return None
        response.raise_for_status()
        rows.extend(response.json()['results'])
    return rows


api_results = fetch_histories(FIELD_IDS) if API_KEY else None
if api_results is None:
    print('Skipping API response parsing until a valid API key is set.')
else:
    print(f'Fetched {len(api_results)} field histories through POST /v1/fields.')


In [ ]:
if not api_results:
    print('Set a valid API key and rerun the previous cell to inspect the response.')
else:
    flat_rows = []
    for item in api_results:
        field_id = item['field_id']
        for entry in item.get('history', []):
            flat_rows.append(
                {
                    'field_id': field_id,
                    'year': entry['year'],
                    'crop_code': entry['crop_code'],
                    'crop_name': entry['crop_name'],
                    'fraction': entry.get('fraction'),
                }
            )

    api_history = (
        pd.DataFrame(flat_rows)
        .merge(field_lookup, on='field_id', how='left')
        .sort_values(['field_id', 'year'])
        .reset_index(drop=True)
    )
    print(f'Flattened {len(api_history):,} history rows from the API response.')
    display(api_history.head(20))


In [ ]:
if not api_results:
    print('The raw API preview will appear here after a successful request.')
else:
    api_results[0]


In [ ]:
if not api_results:
    print('The pivoted history view will appear here after a successful request.')
else:
    pivot = api_history.pivot(index='year', columns='field_id', values='crop_name').sort_index()
    display(pivot)
